In [2]:
import requests
import pandas as pd
import numpy as np

start_date, end_date = "20251201", "20260531"
lat, lon = -7.889008716609542, 112.51617736151893

params = "T2M_MAX,T2M_MIN,RH2M"

url = (
    f"https://power.larc.nasa.gov/api/temporal/daily/point?"
    f"parameters={params}&community=AG&longitude={lon}&latitude={lat}"
    f"&start={start_date}&end={end_date}&format=JSON"
)

print("Fetch data dari NASA POWER API...")
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data['properties']['parameter'])
    df.index = pd.to_datetime(df.index)

    # Buang nilai error/fill value dari NASA (-999.0)
    df = df.replace(-999.0, np.nan).dropna()
    
    # Export ke CSV (Tanpa filter mask karena API sudah memfilter by date)
    df.to_csv("nasa_weather_batu_dec_may.csv")
    print("Data berhasil disimpan ke nasa_weather_batu_dec_may.csv\n")

    print("=== Rekomendasi CFG dari Data Satelit (Telah Disesuaikan ke Mikroklimat Greenhouse) ===")
    
    # --- KONSTANTA GREENHOUSE (DELTA) ---
    # Asumsi: Suhu dalam greenhouse 2 derajat lebih panas dari luar
    # Kelembapan maksimal greenhouse dibatasi 85% untuk mencegah jamur (sesuai literatur paprika)
    DELTA_TEMP = 2.0 
    MAX_RH_GREENHOUSE = 85.0 

    # Kalkulasi Suhu (Ditambah Delta Greenhouse)
    t_puncak = df['T2M_MAX'].mean() + DELTA_TEMP
    t_lembah = df['T2M_MIN'].mean() + DELTA_TEMP
    t_std = (df['T2M_MAX'].std() + df['T2M_MIN'].std()) / 2

    # Kalkulasi Kelembapan 
    h_mean = df['RH2M'].mean()
    h_std = df['RH2M'].std()
    
    # Dibatasi ke batas toleransi agronomis paprika, bukan 100% cuaca luar
    h_puncak = min(h_mean + (h_std * 2), MAX_RH_GREENHOUSE) 
    h_lembah = h_mean - (h_std * 2)

    print(f"TEMP_CFG = {{")
    print(f"    'peak_mean': {t_puncak:.1f},  # Makroklimat NASA + {DELTA_TEMP}C (Greenhouse Effect)")
    print(f"    'trough_mean': {t_lembah:.1f},")
    print(f"    'noise_std': {t_std:.1f},")
    print(f"    'min': {t_lembah - (t_std*3):.1f},")
    print(f"    'max': {t_puncak + (t_std*3):.1f},")
    print(f"    'peak_hour': 13,")
    print(f"    'trough_hour': 4")
    print(f"}}")

    print(f"\nHUM_CFG = {{")
    print(f"    'peak_mean': {h_puncak:.1f},  # Dibatasi max {MAX_RH_GREENHOUSE}% (Pencegahan Botrytis)")
    print(f"    'trough_mean': {h_lembah:.1f},")
    print(f"    'noise_std': {h_std:.1f},")
    print(f"    'min': {h_lembah - (h_std*2):.1f},")
    print(f"    'max': {MAX_RH_GREENHOUSE:.1f},")
    print(f"    'peak_hour': 4,")
    print(f"    'trough_hour': 13")
    print(f"}}")

else:
    print(f"Error fetching data. Status code: {response.status_code}")

Fetch data dari NASA POWER API...
Data berhasil disimpan ke nasa_weather_batu_dec_may.csv

=== Rekomendasi CFG dari Data Satelit (Telah Disesuaikan ke Mikroklimat Greenhouse) ===
TEMP_CFG = {
    'peak_mean': 29.3,  # Makroklimat NASA + 2.0C (Greenhouse Effect)
    'trough_mean': 24.1,
    'noise_std': 0.9,
    'min': 21.5,
    'max': 31.9,
    'peak_hour': 13,
    'trough_hour': 4
}

HUM_CFG = {
    'peak_mean': 85.0,  # Dibatasi max 85.0% (Pencegahan Botrytis)
    'trough_mean': 84.8,
    'noise_std': 2.2,
    'min': 80.3,
    'max': 85.0,
    'peak_hour': 4,
    'trough_hour': 13
}
